In [ ]:
!pip install -q torch transformers==4.46.0 accelerate bitsandbytes timm==0.9.16 flask pyngrok pillow

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"      # stop transformers importing tensorflow
os.environ["TRANSFORMERS_NO_FLAX"] = "1"    # stop transformers importing jax/flax

import base64, torch
from io import BytesIO
from PIL import Image
from flask import Flask, request, jsonify
from pyngrok import ngrok
from transformers import AutoModelForVision2Seq, AutoProcessor
from google.colab import userdata

In [ ]:
MODEL_PATH = "openvla/openvla-7b"

print("[System] Loading OpenVLA...")

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)

vla = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH,
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    load_in_4bit=True,
    trust_remote_code=True,
)

print("[System] Model ready.\n")

In [ ]:
ngrok.set_auth_token(userdata.get('ngrok_auth_token'))

app = Flask(__name__)
SERVER_PORT = 5000
# UNNORM_KEY = "berkeley_autolab_ur5"
UNNORM_KEY = "bridge_orig"

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data        = request.json
        base64_img  = data.get("image")
        instruction = data.get("instruction")

        if not base64_img or not instruction:
            return jsonify({"status": "error", "message": "Missing image or instruction"}), 400

        image = Image.open(BytesIO(base64.b64decode(base64_img))).convert("RGB")
        image = image.resize((224, 224), Image.LANCZOS)

        prompt = f"In: What action should the robot take to {instruction}?\nOut:"

        inputs = processor(prompt, image).to("cuda:0", dtype=torch.bfloat16)
        action = vla.predict_action(**inputs, unnorm_key=UNNORM_KEY, do_sample=False)

        print(f"[Predict] {action.tolist()}")
        return jsonify({"status": "success", "action": action.tolist()})

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"status": "error", "message": str(e)}), 500


tunnel = ngrok.connect(SERVER_PORT)
print(f"[ngrok] {tunnel.public_url}")
print(f'[config.py] VLA_API_URL = "{tunnel.public_url}"')

app.run(port=SERVER_PORT)